# Namma Space: Colab reconstruction (frames -> Gaussian splat .ply)

Mirrors `pipeline/reconstruct.py`. **Not yet run end-to-end; verify on first use.**
Use a GPU runtime (Runtime > Change runtime type > T4/L4/A100).

Input: a zip of extracted frames (`out/frames/*.jpg` from `extract_frames.py`).

In [ ]:
!nvidia-smi
!apt-get -qq install -y colmap ffmpeg  # apt COLMAP may be CPU-only (slower feature extraction)

In [ ]:
!pip -q install nerfstudio  # follow https://docs.nerf.studio/quickstart/installation.html if this fails

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()  # upload frames.zip
os.makedirs('frames', exist_ok=True)
zipfile.ZipFile(list(up)[0]).extractall('frames')
!ls frames | head

In [ ]:
MATCHER = 'sequential'   # 'exhaustive' for quality mode
ITERS = 10000            # fast ~7000, quality ~30000
!ns-process-data images --data frames --output-dir ns_data --matching-method {MATCHER}

In [ ]:
import json
n_in = len([f for f in os.listdir('frames') if f.endswith('.jpg')])
n_reg = len(json.load(open('ns_data/transforms.json'))['frames'])
print(f'Registered {n_reg}/{n_in} = {n_reg/n_in:.0%}')
if n_reg/n_in < 0.8: print('WARNING <80%: walk slower, more overlap, better light (see docs/SOP.md)')

In [ ]:
!ns-train splatfacto --data ns_data --max-num-iterations {ITERS} --output-dir train --viewer.quit-on-train-completion True

In [ ]:
import glob
cfg = sorted(glob.glob('train/**/config.yml', recursive=True))[-1]
!ns-export gaussian-splat --load-config {cfg} --output-dir splat
!ls -la splat

In [ ]:
files.download(glob.glob('splat/*.ply')[0])  # copy into <out>/splat.ply then run the remaining stages locally